In [ ]:
# ==============================================================================
# Reprojecting RGB Digital Surface Model to BNG/ODN
# Transformation: OSTN15 (Horizontal) & OSGM15 (Vertical)
# From: EPSG:32630 (UTM 30N)
# To:   EPSG:27700 (BNG) + ODN Height
# Date: April 2026
# ==============================================================================

In [ ]:
%pip install pyproj rasterio -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Test if the grids are working
import os
import pyproj
from pyproj import Transformer

## Setup Paths
# Local Users: Leave it as BASE_DIR = ""
# Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

# Point PyProj to your local/Drive grids folder
grid_folder = os.path.join(BASE_DIR, "Transformation_Grids")  # Replace "Transformation_Grids" with the name of the folder you kept the grid files
pyproj.datadir.append_data_dir(grid_folder)

# Build the pipeline referencing the .tif and .gsb files
t = Transformer.from_pipeline(
    "+proj=pipeline "
    "+step +inv +proj=utm +zone=30 +ellps=WGS84 "
    "+step +inv +proj=vgridshift +grids=uk_os_osgm15_gb.tif +multiplier=1 "
    "+step +proj=hgridshift +grids=ostn15_etrs_to_osgb.gsb "
    "+step +proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 "
    "+x_0=400000 +y_0=-100000 +ellps=airy +units=m +no_defs"
)

# Transform a test point (UTM Easting, UTM Northing, Ellipsoidal Height)
e_bng, n_bng, z_odn = t.transform(456000.0, 5850000.0, 75.0)

print("Validation Completed!!!")
print(f"BNG:   {e_bng:.3f}, {n_bng:.3f}")
print(f"Z ODN: {z_odn:.3f}")

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import pyproj
from pyproj import Transformer

## 1. DEFINE THE FUNCTION
def reproject_dsm_odn(input_path, output_path, grid_folder):
    print("Setting up transformation pipeline...")

    # PyProj Setup (For the Z-Shift)
    pyproj.datadir.append_data_dir(grid_folder)

    # Rasterio Setup (For the XY-Shift)
    # Build absolute path to the grid for Rasterio
    gsb_path = os.path.abspath(os.path.join(grid_folder, "ostn15_etrs_to_osgb.gsb"))
    bng_2d_pipeline = f"+proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 +x_0=400000 +y_0=-100000 +ellps=airy +units=m +no_defs +nadgrids=@{gsb_path}"

    # Build the Z-shift tool using PyProj
    z_shift_pipeline = (
        "+proj=pipeline "
        "+step +inv +proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 +x_0=400000 +y_0=-100000 +ellps=airy "
        "+step +inv +proj=hgridshift +grids=ostn15_etrs_to_osgb.gsb "
        "+step +inv +proj=vgridshift +grids=uk_os_osgm15_gb.tif +multiplier=1"
    )
    bng_z_shifter = Transformer.from_pipeline(z_shift_pipeline)

    print("Starting rasterio reprojection...")
    with rasterio.open(input_path) as src:
        transform, width, height = calculate_default_transform(
            "EPSG:32630", "EPSG:27700", src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            'crs': 'EPSG:27700',
            'transform': transform,
            'width': width,
            'height': height,
            'nodata': -9999,
            'dtype': 'float32'
        })

        with rasterio.open(output_path, 'w', **kwargs) as dst:
            # Reproject Horizontal Coordinates (XY) using the high-precision string
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs="EPSG:32630",
                dst_transform=transform,
                dst_crs=bng_2d_pipeline,
                src_nodata=src.nodata,
                dst_nodata=-9999,
                resampling=Resampling.bilinear
            )

        print("Applying OSGM15 vertical correction (ODN)...")
        # Apply vertical correction tile-by-tile
        with rasterio.open(output_path, 'r+') as dst:
            for ji, window in dst.block_windows(1):
                dest = dst.read(1, window=window)
                mask = dest != -9999

                if np.any(mask):
                    win_transform = rasterio.windows.transform(window, transform)
                    rows, cols = np.indices(dest.shape)
                    xs = win_transform.c + cols * win_transform.a + rows * win_transform.b
                    ys = win_transform.f + cols * win_transform.d + rows * win_transform.e

                    # Convert Z from Ellipsoidal to ODN
                    _, _, z_odn = bng_z_shifter.transform(xs[mask], ys[mask], dest[mask])

                    dest[mask] = z_odn
                    dst.write(dest, 1, window=window)

            dst.update_tags(VERTICAL_DATUM="ODN", GEOID_MODEL="OSGM15")

    print(f"✅ Success: {output_path}")

## 2. EXECUTE THE SCRIPT
if __name__ == "__main__":

## Setup Paths
# Local Users: Leave it as BASE_DIR = ""
# Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

    BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

    input_file = os.path.join(BASE_DIR, 'RGB_DSM.tif') # Replace "RGB_DSM.tif" with your input file name
    output_file = os.path.join(BASE_DIR, 'RGB_DSM_BNG_ODN.tif') # Replace "RGB_DSM_BNG_ODN.tif" with your preferred output file name
    grid_dir = os.path.join(BASE_DIR, 'Transformation_Grids') # Replace "Transformation_Grids" with the name of the folder you kept the grid files

    reproject_dsm_odn(input_file, output_file, grid_dir)